# Chapter 9 - HyperParameter Tuning with Cross-Validation

## Preparation

In [23]:
import os
import sys
sys.path.append(os.path.abspath(os.path.join('..')))

import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import make_classification
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV
from sklearn.svm import SVC

from utils.data_paths import IVE_TICKS
from utils.sampling_bars import dollar_bar
from utils.cv import PurgedKFold
from utils.log_uniform import log_uniform

%matplotlib inline
plt.style.use('ggplot')
plt.rcParams['figure.figsize'] = 16,6

df = pd.read_parquet(IVE_TICKS)

dollar_bars = dollar_bar(df)

## 1. Using the function getTestData from Chapter 8, form a synthetic dataset of  10,000 observations with 10 features, where 5 are informative and 5 are noise.

In [24]:
def get_test_data(n_features=40, n_informative=10, n_redundant=10, n_samples=10000):
    # generate a random dataset for a classification problem    
    trains_df, labels_df = make_classification(
        n_samples=n_samples, 
        n_features=n_features, 
        n_informative=n_informative, 
        n_redundant=n_redundant, 
        random_state=0, 
        shuffle=False
    )
    # Create a DatetimeIndex using start and periods (not end)
    indices = pd.date_range(
        start=pd.Timestamp.today(), 
        periods=n_samples, 
        freq=pd.tseries.offsets.Minute()
    )
    trains_df = pd.DataFrame(trains_df, index=indices)
    labels_df = pd.Series(labels_df, index=indices).to_frame('bin')
    
    columns = ['I_%s' % i for i in range(n_informative)] + ['R_%s' % i for i in range(n_redundant)]
    columns += ['N_%s' % i for i in range(n_features - len(columns))]
    trains_df.columns = columns

    labels_df['w'] = 1.0 / labels_df.shape[0]
    labels_df['t1'] = pd.Series(labels_df.index, index=labels_df.index)
    return trains_df, labels_df

In [25]:
trains_df, labels_df = get_test_data(n_features=10, n_informative=5, n_redundant=0, n_samples=10000)

In [26]:
labels_df.head()

,bin,w,t1
2026-05-26 21:53:08.810392,0,0.0001,2026-05-26 21:53:08.810392
2026-05-26 21:54:08.810392,0,0.0001,2026-05-26 21:54:08.810392
2026-05-26 21:55:08.810392,0,0.0001,2026-05-26 21:55:08.810392
2026-05-26 21:56:08.810392,0,0.0001,2026-05-26 21:56:08.810392
2026-05-26 21:57:08.810392,0,0.0001,2026-05-26 21:57:08.810392


### a. Use GridSearchCV on 10-fold CV to find the C, gamma optimal  hyperparameters on a SVC with RBF kernel, where  param_grid = {'C':[1E-2,1E-1,1,10,100],'gamma':[1E-2,1E1,1,10,100]} and the scoring function is neg_log_loss.

In [27]:
purged_cv = PurgedKFold(n_splits=10, t1=labels_df['t1'])

features = trains_df
target = labels_df['bin']
sample_weights = labels_df['w']

grid_search = GridSearchCV(
    estimator=SVC(kernel='rbf', probability=True),
    param_grid={'C':[1e-1,10,100],'gamma':[1e1,10,100]},
    scoring='neg_log_loss',
    cv=purged_cv,
    return_train_score=True,
    verbose=1,
    n_jobs=3
)

grid_search.fit(features, target)

Fitting 10 folds for each of 9 candidates, totalling 90 fits


,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",SVC(probability=True)
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'C': [0.1, 10, ...], 'gamma': [10.0, 10, ...]}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'neg_log_loss'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",3
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",PurgedKFold(n...atetime64[us])
,"verbose verbose: intControls the verbosity: the higher, the more messages.- >1 : the computation time for each fold and parameter candidate is displayed;- >2 : the score is als

In [28]:
best_params_gs = grid_search.best_params_
best_score_gs = grid_search.best_score_

grid_search_results = pd.DataFrame(grid_search.cv_results_)

print(f"Best parameters of grid search: {best_params_gs}")
print(f"Best score of grid search: {best_score_gs}")

Best parameters of grid search: {'C': 0.1, 'gamma': 10.0}
Best score of grid search: -0.7884338918109027


### b. How many nodes are there in the grid?

In [29]:
print(f"Number of nodes in the grid: {len(grid_search.param_grid['C']) * len(grid_search.param_grid['gamma'])}")

Number of nodes in the grid: 9


### c. How many fits did it take to find the optimal solution?

In [30]:
print(f"Number of fits: {grid_search_results.shape[0]}")

Number of fits: 9


### d. How long did it take to find this solution?

In [31]:
print(f"It took {grid_search_results['mean_fit_time'].sum():.0f} seconds for grid search.")

It took 76 seconds for grid search.


### e. How can you access the optimal result?

In [32]:
best_estimator = grid_search.best_estimator_

best_estimator

,"C C: float, default=1.0Regularization parameter. The strength of the regularization isinversely proportional to C. Must be strictly positive. The penaltyis a squared l2 penalty. For an intuitive visualization of the effectsof scaling the regularization parameter C, see:ref:`sphx_glr_auto_examples_svm_plot_svm_scale_c.py`.",0.1
,"kernel kernel: {'linear', 'poly', 'rbf', 'sigmoid', 'precomputed'} or callable, default='rbf'Specifies the kernel type to be used in the algorithm. Ifnone is given, 'rbf' will be used. If a callable is given it is used topre-compute the kernel matrix from data matrices; that matrix should bean array of shape ``(n_samples, n_samples)``. For an intuitivevisualization of different kernel types see:ref:`sphx_glr_auto_examples_svm_plot_svm_kernels.py`.",'rbf'
,"degree degree: int, default=3Degree of the polynomial kernel function ('poly').Must be non-negative. Ignored by all other kernels.",3
,"gamma gamma: {'scale', 'auto'} or float, default='scale'Kernel coefficient for 'rbf', 'poly' and 'sigmoid'.- if ``gamma='scale'`` (default) is passed then it uses 1 / (n_features * X.var()) as value of gamma,- if 'auto', uses 1 / n_features- if float, must be non-negative... versionchanged:: 0.22 The default value of ``gamma`` changed from 'auto' to 'scale'.",10.0
,"coef0 coef0: float, default=0.0Independent term in kernel function.It is only significant in 'poly' and 'sigmoid'.",0.0
,"shrinking shrinking: bool, default=TrueWhether to use the shrinking heuristic.See the :ref:`User Guide `.",True
,"probability probability: bool, default=FalseWhether to enable probability estimates. This must be enabled priorto calling `fit`, will slow down that method as it internally uses5-fold cross-validation, and `predict_proba` may be inconsistent with`predict`. Read more in the :ref:`User Guide `.",True
,"tol tol: float, default=1e-3Tolerance for stopping criterion.",0.001
,"cache_size cache_size: float, default=200Specify the size of the kernel cache (in MB).",200
,"class_weight class_weight: dict or 'balanced', default=NoneSet the parameter C of class i to class_weight[i]*C forSVC. If not given, all classes are supposed to haveweight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.",None
,"verbose verbose: bool, default=FalseEnable verbose output. Note that this setting takes advantage of aper-process runtime setting in libsvm that, if enabled, may not workproperly in a multithreaded context.",False


### f. What is the CV score of the optimal parameter combination?

In [33]:
print(f"CV score of the optimal parameter combination: {grid_search.best_score_}")

CV score of the optimal parameter combination: -0.7884338918109027


### g. How can you pass sample weights to the SVC?

In [35]:
best_estimator.fit(features, target, sample_weight=sample_weights)

best_estimator.score(features, target, sample_weight=sample_weights)

0.5002

## 2. Using the same dataset from exercise 1,

### a. Use RandomizedSearchCV on 10-fold CV to find the C, gamma optimal  hyperparameters on an SVC with RBF kernel, where  param_distributions = {‘C’:logUniform(a = 1E-2,b =  1E2),‘gamma’:logUniform(a = 1E-2,b = 1E2)},n_iter = 25 and neg_log_loss is the scoring function.

In [ ]:
purged_cv = PurgedKFold(n_splits=10, t1=labels_df.index.to_series())

param_distributions = {'C': log_uniform(a=1e-2, b=1e2), 'gamma': log_uniform(a=1e-2, b=1e2)}

n_iter = 9
random_search = RandomizedSearchCV(
    estimator=SVC(kernel='rbf', probability=True), 
    param_distributions=param_distributions, 
    scoring='neg_log_loss',
    cv=purged_cv, 
    n_iter=n_iter,
    return_train_score=True,
    verbose=1,
    n_jobs=3
)

random_search = random_search.fit(X=trains_df, y=labels_df['bin'])
random_search_results = pd.DataFrame(random_search.cv_results_)

Fitting 10 folds for each of 9 candidates, totalling 90 fits


### b. How long did it take to find this solution?

In [14]:
print(f"It took {random_search_results['mean_fit_time'].sum():.0f} seconds for random search.")

It took 41 seconds for random search.


### c. Is the optimal parameter combination similar to the one found in exercise  1?

In [15]:
best_params_rs = random_search.best_params_
best_score_rs = random_search.best_score_
best_estimator_rs = random_search.best_estimator_

print(f"Best parameters of random search: {best_params_rs}")
print(f"Best score of random search: {best_score_rs}")

print()

print(f"Best parameters of grid search: {best_params_gs}")
print(f"Best score of grid search: {best_score_gs}")


Best parameters of random search: {'C': np.float64(3.305627535084805), 'gamma': np.float64(0.022147882956834553)}
Best score of random search: -0.32307179178798184

Best parameters of grid search: {'C': 0.1, 'gamma': 10}
Best score of grid search: -0.7884530375637275


### d. What is the CV score of the optimal parameter combination? How does it  compare to the CV score from exercise 1?

In [16]:
print(f"CV score of the optimal parameter combination of random search: {random_search.best_score_}")
print(f"CV score of the optimal parameter combination of grid search: {grid_search.best_score_}")

CV score of the optimal parameter combination of random search: -0.32307179178798184
CV score of the optimal parameter combination of grid search: -0.7884530375637275


## 3. From exercise 1,

### a. Compute the Sharpe ratio of the resulting in-sample forecasts, from point  1.a (see Chapter 14 for a definition of Sharpe ratio).

In [17]:
def sharpe_ratio(r):
    return r.mean() / r.std()

y_pred = best_estimator.predict(trains_df)
bin_returns = labels_df['bin'] * 2 - 1

print("The Sharpe ratio using neg_log_loss as metric to hyper-tune the parameters (Grid Search) is {:.2f}.".format(sharpe_ratio(y_pred * bin_returns)))

The Sharpe ratio using neg_log_loss as metric to hyper-tune the parameters (Grid Search) is 0.00.


### b. Repeat point 1.a, this time with accuracy as the scoring function. Compute the in-sample forecasts derived from the hyper-tuned parameters.

In [18]:
purged_cv = PurgedKFold(n_splits=10, t1=labels_df['t1'])

features = trains_df
target = labels_df['bin']
sample_weights = labels_df['w']

grid_search_accuracy = GridSearchCV(
    estimator=SVC(kernel='rbf', probability=True),
    param_grid={'C':[1e-1,1,10],'gamma':[1e1,1,10]},
    scoring='accuracy',
    cv=purged_cv,
    return_train_score=True,
    verbose=1,
    n_jobs=3
)

grid_search_accuracy.fit(features, target)

Fitting 10 folds for each of 9 candidates, totalling 90 fits


,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",SVC(probability=True)
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'C': [0.1, 1, ...], 'gamma': [10.0, 1, ...]}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'accuracy'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",3
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",PurgedKFold(n...atetime64[us])
,"verbose verbose: intControls the verbosity: the higher, the more messages.- >1 : the computation time for each fold and parameter candidate is displayed;- >2 : the score is also disp

In [19]:
best_estimator_accuracy = grid_search_accuracy.best_estimator_

y_pred = best_estimator_accuracy.predict(trains_df)
bin_returns = labels_df['bin'] * 2 - 1

print("The Sharpe ratio using accuracy as metric to hyper-tune the parameters (Grid Search) is {:.2f}.".format(sharpe_ratio(y_pred * bin_returns)))

The Sharpe ratio using accuracy as metric to hyper-tune the parameters (Grid Search) is 1.00.


### c. What scoring method leads to higher (in-sample) Sharpe ratio?

The scoring method with accuracy leads to higher Sharpe Ratio in the case of Grid Search.

## 4. From exercise 2,

### a. Compute the Sharpe ratio of the resulting in-sample forecasts, from point  2.a.

In [20]:
best_estimator_rs.predict(trains_df)

print("The Sharpe ratio using accuracy as metric to hyper-tune the parameters (Random Search) is {:.2f}.".format(sharpe_ratio(y_pred * bin_returns)))

The Sharpe ratio using accuracy as metric to hyper-tune the parameters (Random Search) is 1.00.


### b. Repeat point 2.a, this time with accuracy as the scoring function.  Compute the in-sample forecasts derived from the hyper-tuned parameters.

In [21]:
purged_cv = PurgedKFold(n_splits=10, t1=labels_df.index.to_series())

n_iter = 9
random_search_accuracy = RandomizedSearchCV(
    estimator=SVC(kernel='rbf', 
    probability=True), 
    param_distributions=param_distributions, 
    scoring='accuracy',
    cv=purged_cv, 
    n_iter=n_iter,
    return_train_score=True,
    verbose=1,
    n_jobs=3
)

random_search_accuracy = random_search_accuracy.fit(X=trains_df, y=labels_df['bin'])
random_search_results_accuracy = pd.DataFrame(random_search_accuracy.cv_results_)

Fitting 10 folds for each of 9 candidates, totalling 90 fits


In [22]:
best_estimator_accuracy_rs = random_search_accuracy.best_estimator_

y_pred = best_estimator_accuracy_rs.predict(trains_df)

print("The Sharpe ratio using accuracy as metric to hyper-tune the parameters (Random Search) is {:.2f}.".format(sharpe_ratio(y_pred * bin_returns)))

The Sharpe ratio using accuracy as metric to hyper-tune the parameters (Random Search) is 0.67.


### c. What scoring method leads to higher (in-sample) Sharpe ratio?

Using negative log loss as the scoring method lead to higher Sharpe Ratio in the case of Random Search.

## 5. Read the definition of log loss, L[Y, P].

### a. Why is the scoring function neg_log_loss defined as the negative log loss, −L[Y, P]?

When we maximize the negative log loss, we are minimizing the log loss itself. Besides, it is desired to maximize the scoring function, just like accuracy.

### b. What would be the outcome of maximizing the log loss, rather than the  negative log loss?

Then we are rewarding the wrong label, since the only way to have a maximized log loss is to have all predictions wrong. Then we will eventually choose the model with the worst performance.

## 6. Consider an investment strategy that sizes its bets equally, regardless of the forecast's confidence. In this case, what is a more appropriate scoring function for hyperparameter tuning, accuracy or cross-entropy loss?

We would prefer accuracy in this case, since accuracy only looks for outcome without confidence, while cross-entropy loss (negative log loss) will use the confidence (probability) for optimization.